<a href="https://colab.research.google.com/github/mushu2843/openCV/blob/Projects/Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import zipfile
import os
new_zip_file_path = '/content/Smple.zip'
new_extract_to_dir = '/content/Ex_data'
os.makedirs(new_extract_to_dir, exist_ok=True)
print(f"Attempting to unzip'{new_zip_file_path}' to '{new_extract_to_dir}'...")
try:
  with zipfile.ZipFile(new_zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(new_extract_to_dir)
  print("Unzipping complete.")
except zipfile.BadZipFile:
  print(f"Error: '{new_zip_file_path}' is not a valid zip file. Please ensure it's a valid .zip archive.")
except FileNotFoundError:
  print(f"Error: zip file not found at '{new_zip_file_path}'. Please upload your 'data.zip' to '/content/'.")
except Exception as e:
  print(f"An unexpected error occurred during unzipping: {e}.")


Attempting to unzip'/content/Smple.zip' to '/content/Ex_data'...
Unzipping complete.


In [12]:
import os

# Define image extensions
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")

# Define the base directory where the new dataset was extracted
base_dataset_dir = '/content/Ex_data'

# Define the specific subfolders to check
subfolders_to_check = {
    'Train_Person': os.path.join(base_dataset_dir, 'Train', 'Person'),
    'Train_Truck': os.path.join(base_dataset_dir, 'Train', 'Truck'),
    'Test_Person': os.path.join(base_dataset_dir, 'Test', 'Person'),
    'Test_Truck': os.path.join(base_dataset_dir, 'Test', 'Truck')
}

print("\nCounting images in specified subfolders:")

for name, path in subfolders_to_check.items():
    if os.path.exists(path) and os.path.isdir(path):
        count = 0
        for file in os.listdir(path):
            if file.lower().endswith(image_extensions):
                count += 1
        print(f"  {name}: {count} images")
    else:
        print(f"  {name}: Folder not found at '{path}'")


Counting images in specified subfolders:
  Train_Person: 19 images
  Train_Truck: 17 images
  Test_Person: 5 images
  Test_Truck: 5 images


In [14]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os

# Define the base directory for the new dataset
base_dataset_dir = '/content/Ex_data'

# Define paths for training and testing data
train_dir = os.path.join(base_dataset_dir, 'Train')
test_dir = os.path.join(base_dataset_dir, 'Test')

# Define parameters for loading the dataset
IMAGE_SIZE = (128, 128) # Ensure this matches the model's expected input
BATCH_SIZE = 32

print(f"Loading training images from: {train_dir}")
# Load the training data
train_ds_new = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=IMAGE_SIZE,
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(f"\nLoading test images from: {test_dir}")
# Load the test data
test_ds_new = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels='inferred',
    label_mode='int',
    image_size=IMAGE_SIZE,
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=False # No need to shuffle test data
)

print("\nDatasets loaded successfully.")
print(f"Training classes found: {train_ds_new.class_names}")
print(f"Test classes found: {test_ds_new.class_names}")

# Ensure class names are consistent
class_names = train_ds_new.class_names
num_classes = len(class_names)


print("\nBuilding CNN Model...")
# Define the CNN model
model_new = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

# Compile the model
model_new.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model_new.summary()


print("\nTraining the CNN Model...")
# Train the model
epochs_new = 15 # You can adjust the number of epochs
history_new = model_new.fit(
    train_ds_new,
    epochs=epochs_new,
    validation_data=test_ds_new
)


print("\nEvaluating the Model on Test Data...")
# Evaluate the model on the test data
loss_new, accuracy_new = model_new.evaluate(test_ds_new)
print(f"Test Loss: {loss_new:.4f}")
print(f"Test Accuracy: {accuracy_new:.4f}")


# Optional: Plotting training history
# import matplotlib.pyplot as plt
# acc = history_new.history['accuracy']
# val_acc = history_new.history['val_accuracy']
# loss = history_new.history['loss']
# val_loss = history_new.history['val_loss']

# epochs_range = range(epochs_new)

# plt.figure(figsize=(12, 8))
# plt.subplot(1, 2, 1)
# plt.plot(epochs_range, acc, label='Training Accuracy')
# plt.plot(epochs_range, val_acc, label='Validation Accuracy')
# plt.legend(loc='lower right')
# plt.title('Training and Validation Accuracy')

# plt.subplot(1, 2, 2)
# plt.plot(epochs_range, loss, label='Training Loss')
# plt.plot(epochs_range, val_loss, label='Validation Loss')
# plt.legend(loc='upper right')
# plt.title('Training and Validation Loss')
# plt.show()


Loading training images from: /content/Ex_data/Train
Found 36 files belonging to 2 classes.

Loading test images from: /content/Ex_data/Test
Found 10 files belonging to 2 classes.

Datasets loaded successfully.
Training classes found: ['Person', 'Truck']
Test classes found: ['Person', 'Truck']

Building CNN Model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,898 (12.61 MB)

 Trainable params: 3,304,898 (12.61 MB)

 Non-trainable params: 0 (0.00 B)


Training the CNN Model...
Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 435ms/step - accuracy: 0.3889 - loss: 1.1653 - val_accuracy: 0.5000 - val_loss: 0.7983
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 266ms/step - accuracy: 0.5278 - loss: 0.6593 - val_accuracy: 0.5000 - val_loss: 0.7438
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 262ms/step - accuracy: 0.5278 - loss: 0.7737 - val_accuracy: 0.5000 - val_loss: 0.7212
Epoch 4/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 398ms/step - accuracy: 0.5278 - loss: 0.7323 - val_accuracy: 0.5000 - val_loss: 0.6954
Epoch 5/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 308ms/step - accuracy: 0.5278 - loss: 0.6906 - val_accuracy: 0.7000 - val_loss: 0.6751
Epoch 6/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 244ms/step - accuracy: 0.8889 - loss: 0.6650 - val_accuracy: 0.5000 - val_loss: 0.6631
Epoch 7/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 298ms/step - accuracy: 0.4722 - loss: 0.6053 - val_accuracy: 0.5000 - val_loss: 0.9650
Epoch 8/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 246ms/step - accuracy: 0.4722 - loss: 0.7581 - val_a